# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Reviewing the FlyRank methodology, I looked at two common claims:

"Content over 2 years old causes traffic drops." The label comes from historical correlation, not a controlled A/B test. The validation design shows a link, but it cannot prove causation.

"High impression decay predicts lost rankings." This label comes from observing rank drops over a short window, which completely ignores yearly seasonality. The validation shows a directional trend, but the methodology is flawed for seasonal content.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Methodology Audit:")
print("- Claim 1 (Age vs Traffic): Correlation is observed, but causation isn't proven without A/B testing.")
print("- Claim 2 (Decay vs Rankings): The validation split doesn't fully account for macro yearly seasonality.")

Methodology Audit:
- Claim 1 (Age vs Traffic): Correlation is observed, but causation isn't proven without A/B testing.
- Claim 2 (Decay vs Rankings): The validation split doesn't fully account for macro yearly seasonality.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

If I use a random split, the model gets an artificially high score because it memorizes client-specific traffic patterns. It's essentially cheating. When I switch to an honest grouped split (grouping by client_id), the performance drops slightly, but it reflects reality much better. The model is forced to learn universal patterns rather than memorizing specific clients.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import precision_score

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df['impressions_90d'] = df['impressions_90d'].fillna(0)
df['sessions_90d'] = df['sessions_90d'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(df['content_age_days'].median())
df['is_declining'] = df['trend_direction'] == 'down'

features = ['impressions_90d', 'sessions_90d', 'content_age_days']
X, y, groups = df[features], df['is_declining'], df['client_id']

# 1. Bad Split (Random - Cheating)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
model_random = RandomForestClassifier(max_depth=5, random_state=42).fit(X_train_r, y_train_r)
bad_precision = precision_score(y_test_r, model_random.predict(X_test_r))

# 2. Honest Split (Grouped by Client)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
model_grouped = RandomForestClassifier(max_depth=5, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
honest_precision = precision_score(y.iloc[test_idx], model_grouped.predict(X.iloc[test_idx]))

print(f"Random Split (Cheating) Precision: {bad_precision:.2%}")
print(f"Grouped Split (Honest) Precision:  {honest_precision:.2%}")

Random Split (Cheating) Precision: 64.79%
Grouped Split (Honest) Precision:  57.16%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I am doing a final check to ensure our target label (trend_direction) or any identifying flags haven't slipped into the training data. The feature set only contains raw impressions, sessions, and age. The correlation check confirms there are no perfect correlations (near 1.0) that would indicate data leakage.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Final Leakage Hunt
print("Final Features used in model:", features)

# Check correlations with the target
correlations = X.copy()
correlations['target_label'] = y
corr_matrix = correlations.corr()['target_label'].drop('target_label')

print("\nCorrelation with target label:")
print(corr_matrix.round(3))

leakage_detected = (corr_matrix.abs() > 0.95).any()
print(f"\nAny leakage detected (>0.95 correlation)? {leakage_detected}")


Final Features used in model: ['impressions_90d', 'sessions_90d', 'content_age_days']

Correlation with target label:
impressions_90d    -0.018
sessions_90d       -0.023
content_age_days   -0.164
Name: target_label, dtype: float64

Any leakage detected (>0.95 correlation)? False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original bold claim: "My ML model accurately predicts exactly which pages will decay next month based on their age and traffic."

Rewritten safe claim: "The model provides a directional decision-support score based on measured historical traffic and observed content age, which helps prioritize the review queue."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("Original: 'My model accurately predicts exactly which pages will decay.'")
print("Rewritten: 'The model provides a directional decision-support score based on measured historical data.'")

Original: 'My model accurately predicts exactly which pages will decay.'
Rewritten: 'The model provides a directional decision-support score based on measured historical data.'


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.